# Cybersecurity Knowledge Assistant Using RAG 
## Author: Chirag Handa
### Goal: To experiment and develop a RAG Assistant using ChromeDB and LangChain to help in Cybersecurity

In [1]:
# Loading a PDF document

from pypdf import PdfReader
import os

def load_pdf(file_path):
    reader = PdfReader(file_path)
    text = ""

    for page in reader.pages:
        content = page.extract_text()
        if content:
            text += content + "\n"

    return text


def load_all_pdfs(folder):
    docs = []
    for file in os.listdir(folder):
        if file.endswith(".pdf"):
            docs.append(load_pdf(os.path.join(folder, file)))
    return docs


docs = load_all_pdfs("../data/raw_docs")
print(len(docs))

Ignoring wrong pointing object 9 0 (offset 0)
Ignoring wrong pointing object 13 0 (offset 0)
Ignoring wrong pointing object 15 0 (offset 0)
Ignoring wrong pointing object 19 0 (offset 0)
Ignoring wrong pointing object 21 0 (offset 0)
Ignoring wrong pointing object 23 0 (offset 0)
Ignoring wrong pointing object 51 0 (offset 0)
Ignoring wrong pointing object 98 0 (offset 0)
Ignoring wrong pointing object 101 0 (offset 0)
Ignoring wrong pointing object 103 0 (offset 0)
Ignoring wrong pointing object 117 0 (offset 0)
Ignoring wrong pointing object 119 0 (offset 0)
Ignoring wrong pointing object 121 0 (offset 0)
Ignoring wrong pointing object 130 0 (offset 0)
Ignoring wrong pointing object 142 0 (offset 0)
Ignoring wrong pointing object 173 0 (offset 0)
Ignoring wrong pointing object 225 0 (offset 0)
Ignoring wrong pointing object 243 0 (offset 0)
Ignoring wrong pointing object 275 0 (offset 0)
Ignoring wrong pointing object 312 0 (offset 0)
incorrect startxref pointer(1)
parsing for Object

4


In [2]:
# Chunking

from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=200
)

chunks = []
for doc in docs:
    chunks.extend(splitter.split_text(doc))

print("Total chunks:", len(chunks))
print(chunks[0][:300])

c:\Users\chira\miniconda3\envs\cyber-rag\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Total chunks: 1675
MITRE ATT&CKÒ: Design and Philosophy  
        
Authors:  Blake E. Strom Andy Applebaum  Doug P. Miller Kathryn C. Nickels  Adam G. Pennington  Cody B. Thomas  Originally Published July 2018  Revised March 2020 
MP180360R1 MITRE PRODUCT


In [ ]:
# Using HuggingFace Embeddings

from langchain_community.embeddings import HuggingFaceBgeEmbeddings

embeddings = HuggingFaceBgeEmbeddings(
    model_name="BAAI/bge-small-en-v1.5"
)

C:\Users\chira\AppData\Local\Temp\ipykernel_26460\407268295.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings import HuggingFaceBgeEmbeddings
C:\Users\chira\AppData\Local\Temp\ipykernel_26460\407268295.py:3: LangChainDeprecationWarning: The class `HuggingFaceBgeEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceBgeEmbeddings(
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6066.02it/s]


In [4]:
# ChromaDB Vector Store

import chromadb

client = chromadb.PersistentClient(path="./chroma_db")

collection = client.get_or_create_collection(
    name="rag_docs"
)

In [5]:
# Storing embeddings in ChromaDB

import uuid

ids = []
embedded_chunks = []

for chunk in chunks:
    ids.append(str(uuid.uuid4()))
    embedded_chunks.append(chunk)

embeddings_list = embeddings.embed_documents(embedded_chunks)

collection.add(
    ids=ids,
    embeddings=embeddings_list,
    documents=embedded_chunks
)

print("Stored in ChromaDB")

Stored in ChromaDB


In [6]:
# Retreival test

def retrieve(query, k=3):
    query_embedding = embeddings.embed_query(query)

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=k
    )

    return results["documents"][0]


test_query = "What is phishing attack?"
docs = retrieve(test_query)

print(docs)

['and to notify IT Security when they see one. There are many platforms that perform \nphishing tests against users to help educate them on different examples, and track \ntheir improvement over time. Crowd-sourcing this knowledge into notifying IT Security \nteams of phishing helps improve the protections and detections of email-based threats.\nSafeguards\nNUMBER TITLE/DESCRIPTION ASSET TYPE SECURITY FUNCTION IG1 IG2 IG3', 'and to notify IT Security when they see one. There are many platforms that perform \nphishing tests against users to help educate them on different examples, and track \ntheir improvement over time. Crowd-sourcing this knowledge into notifying IT Security \nteams of phishing helps improve the protections and detections of email-based threats.\nSafeguards\nNUMBER TITLE/DESCRIPTION ASSET TYPE SECURITY FUNCTION IG1 IG2 IG3', 'and to notify IT Security when they see one. There are many platforms that perform \nphishing tests against users to help educate them on differ

In [7]:
# LLM Response

from langchain_ollama import OllamaLLM

llm = OllamaLLM(
    model="llama3.1"
)

def generate_answer(query, context_docs):
    context = "\n".join(context_docs)
    prompt = f"""
You are an expert assistant.

Use ONLY the context below.

Context:
{context}

Question:
{query}

Answer:
"""

    response = llm.invoke(prompt)

    return response

In [8]:
# Full RAG pipeline test

def rag(query):
    context_docs = retrieve(query)
    answer = generate_answer(query, context_docs)
    return answer, context_docs


answer, context = rag("What is ransomware?")
print("ANSWER:\n", answer)

ANSWER:
 Ransomware is an extortion technique where data is exfiltrated before being encrypted, and the attacker asks for payment to restore the enterprise's data, as well as to keep it from being sold or publicized.


In [9]:
# Latency test

import time

start = time.time()
rag("What is phishing?")
end = time.time()

print("Latency:", end - start)

Latency: 2.200608015060425


In [10]:
# Evaluation dataset

test_data = [
    {
        "question": "What is the purpose of the MITRE ATT&CK framework and how does it help organizations understand adversary behavior?",
        "keywords": [
            "adversary",
            "tactics",
            "techniques",
            "behavior",
            "threat"
        ]
    },
    {
        "question": "How do CIS Critical Security Controls v8 help organizations prioritize cybersecurity defenses and implement effective security safeguards?",
        "keywords": [
        "prioritize",
        "defenses",
        "safeguards",
        "security",
        "implementation"
    ]
    },
    {
        "question": "What are the major security risks identified in the OWASP Top 10 for Large Language Models and how can organizations mitigate them?",
        "keywords": [
            "LLM",
            "risk",
            "prompt",
            "injection",
            "security"
        ]
    }
]

In [11]:
# Simple retreival evaluation

def retrieval_score(docs, keywords):
    text = " ".join(docs).lower()

    hits = 0
    for k in keywords:
        if k.lower() in text:
            hits += 1

    return hits / len(keywords)


results = []

for item in test_data:
    q = item["question"]
    kws = item["keywords"]

    docs = retrieve(q)
    score = retrieval_score(docs, kws)

    results.append((q, score))

results

[('What is the purpose of the MITRE ATT&CK framework and how does it help organizations understand adversary behavior?',
  0.8),
 ('How do CIS Critical Security Controls v8 help organizations prioritize cybersecurity defenses and implement effective security safeguards?',
  0.4),
 ('What are the major security risks identified in the OWASP Top 10 for Large Language Models and how can organizations mitigate them?',
  0.4)]

In [ ]:
# Tabulating output

import pandas as pd

df = pd.DataFrame(results, columns=["question", "retrieval_score"])
df

,question,retrieval_score
0,What is the purpose of the MITRE ATT&CK framew...,0.8
1,How do CIS Critical Security Controls v8 help ...,0.4
2,What are the major security risks identified i...,0.4


In [ ]:
# Including latency to the results

import time
import pandas as pd

def evaluate_rag(test_data):
    results = []

    for item in test_data:
        question = item["question"]
        expected_keywords = item["keywords"]

        start = time.time()

        retrieved_docs = retrieve(question)

        answer = rag(question)[0]

        latency = time.time() - start

        retrieved_text = " ".join(retrieved_docs).lower()

        hits = 0

        for keyword in expected_keywords:
            if keyword.lower() in retrieved_text:
                hits += 1

        retrieval_score = hits / len(expected_keywords)

        results.append({
            "question": question,
            "answer": answer,
            "retrieval_score": retrieval_score,
            "latency_seconds": latency
        })

    return pd.DataFrame(results)

evaluation_results = evaluate_rag(test_data)

evaluation_results

,question,answer,retrieval_score,latency_seconds
0,What is the purpose of the MITRE ATT&CK framew...,The purpose of the MITRE ATT&CK framework is t...,0.8,2.750897
1,How do CIS Critical Security Controls v8 help ...,According to the context provided by CISecurit...,0.4,5.655844
2,What are the major security risks identified i...,"According to the provided context, the OWASP T...",0.4,10.024161


In [ ]:
# Using Reranker for better retrieval score

from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 10449.68it/s]


In [ ]:
# Reranked retrieval 

def retrieve_with_rerank(query, k=3, fetch_k=10):

    query_embedding = embeddings.embed_query(query)

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=fetch_k
    )

    docs = results["documents"][0]

    pairs = [[query, doc] for doc in docs]

    scores = reranker.predict(pairs)

    scored_docs = list(zip(docs, scores))
    scored_docs.sort(key=lambda x: x[1], reverse=True)

    top_docs = [doc for doc, score in scored_docs[:k]]

    return top_docs

In [ ]:
# RAG pipeline

def rag(query):
    context_docs = retrieve_with_rerank(query)
    answer = generate_answer(query, context_docs)
    return answer, context_docs


answer, context = rag("What is ransomware?")
print("ANSWER:\n", answer)

ANSWER:
 Ransomware is a type of cyber threat that encrypts an enterprise's data and demands ransom for its restoration.


In [ ]:
# Retreival evaluation

def retrieval_score(docs, keywords):
    text = " ".join(docs).lower()

    hits = 0
    for k in keywords:
        if k.lower() in text:
            hits += 1

    return hits / len(keywords)


results = []

for item in test_data:
    q = item["question"]
    kws = item["keywords"]

    docs = retrieve_with_rerank(q)
    score = retrieval_score(docs, kws)

    results.append((q, score))

results

[('What is the purpose of the MITRE ATT&CK framework and how does it help organizations understand adversary behavior?',
  0.8),
 ('How do CIS Critical Security Controls v8 help organizations prioritize cybersecurity defenses and implement effective security safeguards?',
  0.4),
 ('What are the major security risks identified in the OWASP Top 10 for Large Language Models and how can organizations mitigate them?',
  0.6)]

In [ ]:
# Final results with reranking

import time
import pandas as pd

def evaluate_rag(test_data):
    results = []

    for item in test_data:
        question = item["question"]
        expected_keywords = item["keywords"]

        start = time.time()

        retrieved_docs = retrieve_with_rerank(question)

        answer = rag(question)[0]

        latency = time.time() - start

        retrieved_text = " ".join(retrieved_docs).lower()

        hits = 0

        for keyword in expected_keywords:
            if keyword.lower() in retrieved_text:
                hits += 1

        retrieval_score = hits / len(expected_keywords)

        results.append({
            "question": question,
            "answer": answer,
            "retrieval_score": retrieval_score,
            "latency_seconds": latency
        })

    return pd.DataFrame(results)

evaluation_results = evaluate_rag(test_data)

evaluation_results

,question,answer,retrieval_score,latency_seconds
0,What is the purpose of the MITRE ATT&CK framew...,The primary purpose of the MITRE ATT&CK framew...,0.8,3.457927
1,How do CIS Critical Security Controls v8 help ...,According to the CISecurity.org website (https...,0.4,4.884125
2,What are the major security risks identified i...,"Based on the context provided, I will extract ...",0.6,12.384369
